# 3.3 LangChain Ecosystem Overview

## Playground Notebook

In this notebook, we'll explore the **LangChain ecosystem** hands-on using a **local Ollama model**.

| Topic | What You'll Learn |
|-------|-------------------|
| **What is LangChain** | The framework, its purpose, and the problem it solves |
| **The Five Pillars** | Core, LangChain, Community/Partners, LangGraph, LangSmith |
| **Core Building Blocks** | Models, Prompts, Chains, Output Parsers |
| **LCEL (Pipe Syntax)** | Composing components with the `\|` operator |
| **Memory** | Conversation history management |
| **Swappability** | How LangChain abstractions let you swap providers easily |

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [ ]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama langchain-community

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick test — make sure Ollama is running
test = llm.invoke("Say 'hello' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Test response: {test.content}")

c:\Users\shiva\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Connected to Ollama | Model: llama3.2:3b
   Test response: Hello


---

## 1. Why LangChain? — The Problem It Solves

LLMs by themselves are **stateless text generators**. To build real applications you need:

- Prompt management
- Chaining multiple reasoning steps
- Connecting to external data
- Conversation memory
- Tool orchestration
- Error handling

**Without LangChain** — a Document Q&A system requires 1000+ lines of custom, tightly coupled code.

**With LangChain** — the same system can be built with ~30 lines of composable, modular code.

### The Five Pillars of the LangChain Ecosystem

| Pillar | Package | Purpose |
|--------|---------|----------|
| **1. Core** | `langchain-core` | Base abstractions, Runnable protocol, LCEL, message types |
| **2. LangChain** | `langchain` | Chains, agents, memory modules — cognitive architectures |
| **3. Community/Partners** | `langchain-ollama`, etc. | 700+ integrations — LLM providers, vector DBs, tools |
| **4. LangGraph** | `langgraph` | Stateful agents with graph-based control flow |
| **5. LangSmith** | `langsmith` | Observability — debugging, testing, monitoring |

Let's explore the most important ones hands-on.

---

## 2. Pillar 1 — LangChain Core: Message Types & the Runnable Protocol

Everything in LangChain implements the **Runnable protocol** — a universal interface with:

| Method | What It Does |
|--------|--------------|
| `invoke()` | Process a single input |
| `stream()` | Get output token by token |
| `batch()` | Process multiple inputs at once |

LangChain uses structured **message types** instead of raw dicts:

```
SystemMessage   →  Sets behavior, persona, constraints
HumanMessage    →  The user's input
AIMessage       →  The model's response
```

### Experiment 2A: Using Message Types Directly

In [2]:
# LangChain message types — structured, typed, and self-documenting
messages = [
    SystemMessage(content="You are a concise technical explainer. Max 2 sentences only."),
    HumanMessage(content="What is LangChain?")
]

response = llm.invoke(messages)

print(f"Type: {type(response).__name__}")
print(f"Content: {response.content}")

Type: AIMessage
Content: LangChain is an open-source Python library that provides a unified interface for building and composing various natural language processing (NLP) tasks, such as text classification, sentiment analysis, and question answering, using popular transformer-based architectures like BERT, RoBERTa, and XLNet.


### Experiment 2B: The Runnable Protocol — `invoke()`, `stream()`, `batch()`

In [3]:
# Every LangChain component supports these three methods

# --- invoke: single input, full response ---
print("=" * 50)
print("invoke() — Single input, full response")
print("=" * 50)
result = llm.invoke("What is Python? One sentence.")
print(result.content)

invoke() — Single input, full response
Python is a high-level, interpreted programming language that facilitates rapid development, flexibility, and ease of use, often used for web development, data analysis, machine learning, automation, and more.


In [4]:
# --- stream: token by token ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)

for chunk in llm.stream("Name 50 programming languages. Brief."):
    print(chunk.content, end="", flush=True)
print()

stream() — Tokens arrive as generated
Here are 50 programming languages, briefly described:

1. **Java**: Object-oriented language for Android app development.
2. **Python**: General-purpose language for web development, data analysis, and machine learning.
3. **JavaScript**: Scripting language for client-side web development and server-side programming.
4. **C++**: High-performance language for game development, system programming, and high-performance computing.
5. **C#**: Modern, object-oriented language for Windows app development and mobile app development.
6. **Ruby**: Dynamic language for web development with the Ruby on Rails framework.
7. **Swift**: Language for developing iOS, macOS, watchOS, and tvOS apps.
8. **Go**: Concurrent and parallel programming language developed by Google.
9. **Rust**: Systems programming language that prioritizes safety and performance.
10. **PHP**: Server-side scripting language for web development.
11. **TypeScript**: Superset of JavaScript for b

In [5]:
# --- batch: multiple inputs at once ---
print("=" * 50)
print("batch() — Process multiple inputs")
print("=" * 50)

questions = [
    "What is an API? One sentence.",
    "What is a vector database? One sentence.",
    "What is RAG? One sentence."
]

results = llm.batch(questions)

for q, r in zip(questions, results):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")

batch() — Process multiple inputs
Q: What is an API? One sentence.
A: An API, or Application Programming Interface, is a set of defined rules and protocols that allow different software systems to communicate with each other, enabling data exchange, functionality reuse, and integration across various platforms and applications.

Q: What is a vector database? One sentence.
A: A vector database is a type of database that stores and manages vectors, which are mathematical representations of objects or entities, allowing for efficient querying and retrieval of similar items based on their semantic similarity.

Q: What is RAG? One sentence.
A: RAG (Relative Attention Graph) is a neural network architecture that uses relative attention to generate better contextualized representations of input sequences, particularly useful for tasks like question answering and text classification.



---

## 3. Prompt Templates — Reusable, Parameterized Prompts

Instead of building prompt strings manually, LangChain provides **ChatPromptTemplate** — a reusable template with variables.

```python
# Manual string building (fragile, error-prone)
prompt = f"Explain {topic} to a {audience}"

# LangChain template (reusable, composable)
template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}."),
    ("human", "Explain {topic} to a {audience}.")
])
```

### Experiment 3A: Creating and Using Prompt Templates

In [6]:
# Define a reusable prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Keep answers to 2-3 sentences."),
    ("human", "Explain {topic} to a {audience}.")
])

# Format with specific values
messages = prompt.invoke({
    "role": "friendly teacher",
    "topic": "LangChain",
    "audience": "beginner programmer"
})

print(messages)

# See what was generated
print("Formatted messages:")
for msg in messages.messages:
    print(f"  [{type(msg).__name__}] {msg.content}")

print("\nResponse:")
response = llm.invoke(messages)
display(Markdown(response.content))

messages=[SystemMessage(content='You are a friendly teacher. Keep answers to 2-3 sentences.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain LangChain to a beginner programmer.', additional_kwargs={}, response_metadata={})]
Formatted messages:
  [SystemMessage] You are a friendly teacher. Keep answers to 2-3 sentences.
  [HumanMessage] Explain LangChain to a beginner programmer.

Response:


LangChain is an open-source library that allows developers to build and compose AI-powered workflows using natural language processing (NLP) models. Think of it like building with LEGO blocks, but instead of physical blocks, you're building with code!

In simple terms, LangChain provides a way for you to:

1. **Define tasks**: You define what task you want to accomplish, such as sentiment analysis or text classification.
2. **Choose models**: You select the NLP model that best fits your task, such as BERT or RoBERTa.
3. **Compose workflows**: You use LangChain's APIs to create a workflow that combines multiple tasks and models in a specific order.

LangChain is particularly useful for:

* **Automating tedious tasks**: By automating repetitive tasks, you can focus on more strategic work.
* **Building scalable workflows**: LangChain helps you build workflows that can handle large volumes of data and scale with your needs.

Imagine you're building an AI-powered chatbot. You could use LangChain to:

1. Define the task: "Take a user's input and classify it as positive, negative, or neutral."
2. Choose models: Select a pre-trained BERT model for sentiment analysis.
3. Compose workflows: Use LangChain to create a workflow that:
	* Takes user input
	* Passes it through the BERT model
	* Returns the predicted sentiment

LangChain provides a simple and intuitive API, making it easy for developers to build complex AI-powered workflows without requiring extensive NLP expertise.

In [7]:
# Reuse the SAME template with different inputs
scenarios = [
    {"role": "scientist", "topic": "vector embeddings", "audience": "high school student"},
    {"role": "chef", "topic": "APIs", "audience": "someone who only knows cooking"},
]

for scenario in scenarios:
    print(f"\n{'=' * 50}")
    print(f"Role: {scenario['role']} | Topic: {scenario['topic']}")
    print(f"{'=' * 50}")
    messages = prompt.invoke(scenario)
    response = llm.invoke(messages)
    display(Markdown(response.content))


Role: scientist | Topic: vector embeddings


Vector embeddings are a way to represent words, objects, or ideas as mathematical vectors (or lines) that can be used by computers.

Think of it like this: Imagine you're at a music festival, and you want to describe the sound of your favorite band's song. You could try to use lots of adjectives like "energetic," "upbeat," and "catchy." But these words are just simple labels that don't really capture the essence of the song.

Vector embeddings are like a way to take those adjectives and turn them into a special kind of math problem that computers can understand. You're essentially creating a unique "fingerprint" for each word or concept, where each dimension (or axis) in the vector represents something about that idea.

For example, if you wanted to represent the word "dog" as a vector, it might look like this:

[1, 0, 0, ...]

This means that the dog is associated with the first dimension (let's say "animal"), and not with any of the other dimensions. But if you wanted to represent the word "cat", it might look like this:

[0, 1, 0, ...]

This means that the cat is more closely associated with the second dimension (also an animal), but not as strongly as the dog.

The idea behind vector embeddings is that by using these mathematical representations, computers can perform tasks like pattern recognition, classification, and clustering much more efficiently than they could with just simple words or labels.


Role: chef | Topic: APIs


Imagine you're running a busy restaurant, and you have many different dishes that need to be prepared for your customers. To make things easier, you decide to hire some sous chefs (assistants) to help you with the orders.

These sous chefs can receive instructions from you through a "recipe book" (a set of rules or guidelines). When a customer places an order, the sous chef checks the recipe book to see what ingredients are needed and how to prepare them. They then follow the instructions in the book to create the dish.

An API is like a super-advanced recipe book that allows computers to "talk" to each other and exchange information. Just like how your sous chefs need access to your recipe book, an application (like a website or mobile app) needs to be able to communicate with an API to get the information it needs.

Here's how it works:

1. **Request**: The application sends a request to the API, saying "Hey, I need some information about this specific dish." This is like asking your sous chef to prepare a particular order.
2. **Response**: The API receives the request and says "Okay, here's what you need for that dish." It then provides the necessary information (like a list of ingredients or cooking instructions) back to the application.
3. **Action**: The application takes the information from the API and uses it to perform an action (like preparing the dish).

APIs are like a centralized hub that makes it easy for applications to get the information they need, just like how your sous chefs rely on you to provide them with the recipe book.

---

## 4. LCEL — LangChain Expression Language (The Pipe Operator)

LCEL is LangChain's **declarative composition syntax**. You chain components together using the pipe `|` operator — just like Unix pipes.

```
prompt | model | output_parser
```

Each component takes input from the previous one:

```
{variables} → PromptTemplate → Messages → ChatModel → AIMessage → StrOutputParser → string
```

The result is a **chain** — a new Runnable that supports `invoke()`, `stream()`, and `batch()`.

### Experiment 4A: Building Your First Chain with LCEL

In [8]:
# Three components snapped together with |
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}")
])

parser = StrOutputParser()  # Extracts just the text from AIMessage

# Build the chain
chain = prompt | llm | parser

print(f"Chain type: {type(chain).__name__}")
print(f"Chain components: PromptTemplate | ChatOllama | StrOutputParser")

# invoke() — just pass the variables
result = chain.invoke({"question": "What are the 5 pillars of LangChain?"})

print(f"\nResult type: {type(result).__name__}")  # str, not AIMessage!
display(Markdown(result))

Chain type: RunnableSequence
Chain components: PromptTemplate | ChatOllama | StrOutputParser

Result type: str


LangChain is an open-source framework developed by Meta AI that provides a set of tools to build flexible, scalable, and modular applications for natural language processing (NLP). The 5 pillars of LangChain are:

1. **Transformers**: LangChain leverages popular transformer-based architectures like BERT, RoBERTa, and XLNet to perform various NLP tasks such as text classification, sentiment analysis, and question answering.
2. **Tokenization**: LangChain provides efficient tokenization pipelines using libraries like Hugging Face's `tokenizers` and `transformers`. This enables fast and accurate processing of large amounts of text data.
3. **Data Loading and Preprocessing**: The framework offers flexible data loading and preprocessing capabilities, allowing users to easily load, preprocess, and manipulate their own datasets.
4. **Model Selection and Management**: LangChain provides a simple way to select and manage pre-trained models using the `langchain.datasets` library. This enables users to easily switch between different models or fine-tune existing ones for specific tasks.
5. **Task Generation**: The framework offers a variety of task generation tools, including `langchain.tasks`, which allows users to create custom tasks by combining different components and modules. This enables developers to build domain-specific applications tailored to their needs.

By providing these 5 pillars, LangChain empowers developers to build robust, scalable, and modular NLP applications with ease.

### Experiment 4B: Streaming Through a Chain

Because every component implements the Runnable protocol, the entire chain supports streaming automatically.

In [9]:
# Streaming works through the entire chain
print("Streaming through the chain:\n")

for chunk in chain.stream({"question": "What is LCEL in LangChain? Brief explanation."}):
    print(chunk, end="", flush=True)

print()

Streaming through the chain:

In the context of LangChain, **LCE** (Long Context Embedding) refers to a way of representing text as a numerical embedding that captures the long-range dependencies and relationships within a sentence or document.

In traditional language models like BERT and RoBERTa, the input is typically a sequence of tokens (e.g., words or subwords), which are then processed through multiple layers of transformer encoder. However, this approach may not capture the long-range contextual information effectively, especially for longer sequences.

LCE addresses this limitation by using a novel embedding scheme that breaks down the input into smaller chunks and represents each chunk as a separate embedding. These embeddings are then combined to form a final representation of the entire input sequence.

The main benefits of LCE in LangChain include:

* Improved handling of long-range dependencies
* Better capture of contextual relationships between words
* Increased efficie

### Experiment 4C: A More Useful Chain — Topic Explainer

In [10]:
# A reusable chain with multiple template variables
explainer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} instructor. Explain concepts clearly in {length}."),
    ("human", "Explain: {concept}")
])

explainer_chain = explainer_prompt | llm | StrOutputParser()

# Use it for different topics
topics = [
    {"domain": "AI/ML", "concept": "the Runnable protocol in LangChain", "length": "3 sentences"},
    {"domain": "software engineering", "concept": "why modular architecture matters", "length": "2 sentences"},
]

for topic in topics:
    print(f"\n{'=' * 50}")
    print(f"Concept: {topic['concept']}")
    print(f"{'=' * 50}")
    result = explainer_chain.invoke(topic)
    display(Markdown(result))


Concept: the Runnable protocol in LangChain


**The Runnable Protocol in LangChain**

In the context of LangChain, a Python library for building scalable and composable language models, the Runnable Protocol is a fundamental concept that enables the creation of executable, modular, and reusable code blocks.

**What is the Runnable Protocol?**

The Runnable Protocol is an interface that defines a contract for objects that can be executed as standalone functions. A Runnable object implements this protocol by providing methods that allow it to perform a specific task or set of tasks. These tasks may involve complex computations, data processing, or other forms of execution.

**Key Characteristics of the Runnable Protocol**

The following are the essential properties of an object that conforms to the Runnable Protocol:

1. **`__call__()` method**: This is the entry point for executing a Runnable object. It should accept some input arguments (if any) and return a value.
2. **`execute()` method**: This method is optional, but it provides a standardized way of executing a Runnable object without passing in specific inputs. If implemented, `execute()` should return the result of execution.
3. **`inputs()` method**: This method returns the expected input(s) for a given Runnable object. It's used to determine what data or parameters are required to execute the code block.
4. **`output_type` attribute**: This attribute specifies the type of output expected from the `execute()` method.

**Benefits of the Runnable Protocol**

The Runnable Protocol offers several advantages in LangChain:

1. **Modularity and Reusability**: By defining a standardized interface for executable code blocks, developers can create modular, reusable components that can be easily combined to build complex workflows.
2. **Decoupling**: The protocol helps decouple execution logic from the input/output data structures, making it easier to reason about and compose code blocks.
3. **Flexibility**: Runnables can be designed to accept various types of inputs, outputs, or even interact with other components via events.

**Example: Implementing a Runnable in LangChain**

Here's an example implementation of a simple `Calculator` Runnable:
```python
import langchain

class Calculator:
    def __init__(self):
        self.memory = 0

    def __call__(self, num1, num2):
        result = num1 + num2
        self.memory += result
        return result

    def execute(self):
        print("Executing calculator...")
        result = self(num1=5, num2=3)
        return result

    @property
    def inputs(self):
        return (int, int)

    @property
    def output_type(self):
        return float
```
In this example, the `Calculator` class implements the Runnable Protocol by providing an implementation of the `__call__()` method. The `execute()` method is optional but provides a standardized way to execute the calculator without passing in specific inputs.


Concept: why modular architecture matters


Modular architecture is a fundamental concept in software engineering, and it has numerous benefits that can significantly impact the development, maintenance, and scalability of software systems.

**Why Modular Architecture Matters?**

Here are some reasons why modular architecture is crucial:

1. **Easier Maintenance**: A modular system makes it easier to maintain and update individual components without affecting the entire system. If a module needs to be changed or replaced, only that specific component needs to be updated, rather than the entire system.
2. **Improved Scalability**: Modular systems can scale more easily, as new modules can be added to handle increased traffic or demand. This allows developers to add features and functionality without having to rewrite large portions of code.
3. **Reduced Coupling**: In a modular system, each module is designed to be independent and self-contained, reducing coupling between components. This makes it easier to change one component without affecting others.
4. **Increased Flexibility**: Modular systems offer more flexibility in terms of customization and integration with other systems. Developers can easily swap out or replace modules to adapt to changing requirements.
5. **Easier Testing**: With a modular system, individual components can be tested independently, reducing the complexity of testing and allowing for more efficient testing strategies.
6. **Improved Code Reusability**: Modular architecture encourages code reuse by breaking down large systems into smaller, independent components that can be reused across different projects or applications.
7. **Better Communication**: When developers work with modular systems, they tend to communicate more effectively about the interfaces and dependencies between modules, leading to better-designed systems.

**Benefits in Practice**

Modular architecture has numerous benefits in practice:

* It allows for faster development and deployment of new features and functionality.
* It reduces downtime and maintenance costs by minimizing the impact of changes on existing codebases.
* It increases the overall reliability and stability of software systems.
* It enables more efficient use of resources, such as memory and processing power.

**Best Practices**

To get the most out of modular architecture:

1. **Keep modules small and focused**: Aim for a single responsibility per module to minimize complexity.
2. **Use clear and consistent naming conventions**: Make sure interfaces and dependencies are well-defined and easy to understand.
3. **Implement robust testing strategies**: Ensure that individual components can be tested independently before integrating them into larger systems.

By adopting modular architecture, developers can create more maintainable, scalable, and flexible software systems that meet the changing needs of users and business requirements.

---

## 5. Pillar 2 — Chains & Memory

LangChain provides **memory modules** to manage conversation history, since LLMs have no built-in memory.

| Memory Type | How It Works |
|-------------|--------------|
| **ConversationBufferMemory** | Stores the full conversation history |
| **ConversationBufferWindowMemory** | Keeps only the last K exchanges |
| **ConversationSummaryMemory** | LLM-generated summary of the conversation |

### Experiment 5A: Manual Conversation Memory

The simplest approach — manage the message list yourself.

In [11]:
MODEL = "qwen2.5:1.5b"

llm = ChatOllama(model=MODEL, temperature=0.7)

In [12]:
# Manual conversation memory — you control the message list
conversation = [
    SystemMessage(content="You are a helpful AI tutor. Keep answers to 1-2 sentences.")
]

def chat_turn(user_input):
    """Add user message, get response, save to history."""
    conversation.append(HumanMessage(content=user_input))
    response = llm.invoke(conversation)
    conversation.append(response)  # AIMessage saved to history
    return response.content

# Multi-turn conversation
turns = [
    "What is LangChain?",
    "What problem does it solve?",
    "What was my first question? provide as it is"
]

for i, user_msg in enumerate(turns, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}")
    print(f"{'=' * 50}")
    print(f"User: {user_msg}")
    answer = chat_turn(user_msg)
    print(f"AI:   {answer}")

print(f"\nTotal messages in history: {len(conversation)}")
print("The model remembers context because we pass the full history each time!")


Turn 1
User: What is LangChain?
AI:   LangChain is an open-source, high-performance library for building large language models in Python and JavaScript using the Hugging Face Transformers framework. It simplifies the process of initializing, training, and deploying LLMs with minimal code changes from existing pipelines.

Turn 2
User: What problem does it solve?
AI:   LangChain addresses the complexity and inefficiency of developing and deploying large language models (LLMs) by providing a streamlined solution using Python and JavaScript libraries like Hugging Face Transformers. This makes it easier for developers to create, train, and deploy LLMs without extensive coding knowledge.

Turn 3
User: What was my first question? provide as it is
AI:   Your first question was: "What is LangChain?"

Total messages in history: 7
The model remembers context because we pass the full history each time!


### Experiment 5B: Conversation With a Window (Last K Turns)

In production, you can't grow the message list forever. A simple solution: keep only the last K exchanges.

In [13]:
def windowed_chat(messages_history, user_input, system_prompt, window_size=2):
    """Chat with a sliding window — keep only last K exchanges."""
    messages_history.append(HumanMessage(content=user_input))

    # Build the windowed context: system + last K pairs
    windowed = [SystemMessage(content=system_prompt)]
    # Keep last window_size * 2 messages (each exchange = human + ai)
    windowed.extend(messages_history[-(window_size * 2 + 1):])

    response = llm.invoke(windowed)
    messages_history.append(response)
    return response.content


# Test it — the model should "forget" early turns
history = []
system = "You are a helpful assistant. Keep answers to 1 sentence."

questions = [
    "My favorite color is blue. Remember that.",
    "What is 2 + 2?",
    "What is the capital of France?",
    "What is my favorite color?"  # Window=2 means this is likely forgotten
]

for i, q in enumerate(questions, 1):
    print(f"\nTurn {i}: {q}")
    print(f"History:{history}")
    answer = windowed_chat(history, q, system, window_size=2)
    print(f"AI:     {answer}")

print("\nWith window_size=2, the model only sees the last 2 exchanges.")
print("Earlier context (like favorite color) may be forgotten!")


Turn 1: My favorite color is blue. Remember that.
History:[]
AI:     Okay, I'll remember that you like blue as your favorite color!

Turn 2: What is 2 + 2?
History:[HumanMessage(content='My favorite color is blue. Remember that.', additional_kwargs={}, response_metadata={}), AIMessage(content="Okay, I'll remember that you like blue as your favorite color!", additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-06-23T15:14:32.5920331Z', 'done': True, 'done_reason': 'stop', 'total_duration': 541969300, 'load_duration': 89213800, 'prompt_eval_count': 35, 'prompt_eval_duration': 195046900, 'eval_count': 15, 'eval_duration': 230416600, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--019ef50c-1bf0-7692-8f45-26f5ad4f4ac1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 15, 'total_tokens': 50})]
AI:     2 + 2 equals 4.

Turn 3: What is the capital of France?
History:[HumanMes

---

## 6. Pillar 3 — Swappability: The Power of Abstractions

LangChain's biggest value: **all providers implement the same interface**.

Switching from OpenAI to Ollama (or Claude, Gemini, etc.) requires changing **one line** — your chain logic stays the same.

```python
# Switch providers — chain logic is IDENTICAL
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_anthropic import ChatAnthropic

llm = ChatOpenAI(model="gpt-4o-mini")       # OpenAI
llm = ChatOllama(model="llama3.2:3b")        # Ollama (local)
llm = ChatAnthropic(model="claude-3-sonnet")  # Anthropic

# Same chain works with ANY of the above
chain = prompt | llm | parser
```

### Experiment 6A: Same Chain, Different Model Configs

In [14]:
# Demonstrate swappability with different Ollama configurations
# (Same concept applies when swapping between OpenAI, Anthropic, etc.)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Explain what a 'chain' is in LangChain. One sentence.")
])
parser = StrOutputParser()

# Config 1: Creative (high temperature)
creative_llm = ChatOllama(model=MODEL, temperature=1.0)
creative_chain = prompt | creative_llm | parser

# Config 2: Precise (low temperature)
precise_llm = ChatOllama(model=MODEL, temperature=0.0)
precise_chain = prompt | precise_llm | parser

print("SAME chain structure, DIFFERENT model configs:\n")

print("Creative (temp=1.0):")
print(creative_chain.invoke({}))

print("\nPrecise (temp=0.0):")
print(precise_chain.invoke({}))

print("\nThe chain logic (prompt | llm | parser) is identical.")
print("Only the model configuration changed!")

SAME chain structure, DIFFERENT model configs:

Creative (temp=1.0):
In LangChain, a chain refers to the sequence of instructions and functions that will be applied to process text data for downstream tasks like summarization or translation.

Precise (temp=0.0):
In LangChain, a chain refers to a sequence of operations that transform input data into output data through a series of functions or modules.

The chain logic (prompt | llm | parser) is identical.
Only the model configuration changed!


---

## 7. Output Parsers — Structured Output from LLMs

LLMs return raw text. **Output parsers** convert that text into structured data.

| Parser | Output |
|--------|---------|
| `StrOutputParser` | Plain string (strips the AIMessage wrapper) |
| `JsonOutputParser` | Parsed JSON dict |
| `CommaSeparatedListOutputParser` | Python list from comma-separated text |

### Experiment 7A: StrOutputParser vs Raw Response

In [15]:
# Without parser — you get an AIMessage object
raw_chain = prompt | llm
raw_result = raw_chain.invoke({})
print(f"Without parser: {type(raw_result).__name__}")
print(f"  Content: {raw_result.content}")

print()

# With parser — you get a clean string
parsed_chain = prompt | llm | StrOutputParser()
parsed_result = parsed_chain.invoke({})
print(f"With StrOutputParser: {type(parsed_result).__name__}")
print(f"  Content: {parsed_result}")

Without parser: AIMessage
  Content: In LangChain, a chain refers to a sequence of operations that transform input data into output results through a series of functions or models.

With StrOutputParser: str
  Content: In LangChain, a chain refers to a sequence of operations that process input data through various model types and components for downstream tasks such as text generation or summarization.


### Experiment 7B: Comma-Separated List Parser

In [16]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()

list_prompt = ChatPromptTemplate.from_messages([
    ("system", "You list items separated by commas. No numbering, no extra text."),
    ("human", "List 5 {category}.")
])

list_chain = list_prompt | llm | list_parser

result = list_chain.invoke({"category": "programming languages"})

print(f"Type: {type(result).__name__}")  # list!
print(f"Result: {result}")
print(f"First item: {result[0]}")
print(f"Count: {len(result)}")

Type: list
Result: ['Python', 'Java', 'C++', 'JavaScript', 'C#']
First item: Python
Count: 5


### Experiment 7C: JSON Output Parser

In [17]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data assistant. Always respond with valid JSON only. "
     "No markdown, no explanation — just the JSON object."),
    ("human",
     "Give me info about {topic}. "
     'Return JSON with keys: "name", "category", "description" (1 sentence).')
])

json_chain = json_prompt | llm | json_parser

result = json_chain.invoke({"topic": "LangChain"})

print(f"Type: {type(result).__name__}")  # dict!
print(f"Result: {result}")

if isinstance(result, dict):
    for key, value in result.items():
        print(f"  {key}: {value}")

Type: dict
Result: {'name': 'LangChain', 'category': 'AI Framework', 'description': 'A flexible, Python-based framework for building and deploying AI models across various platforms.'}
  name: LangChain
  category: AI Framework
  description: A flexible, Python-based framework for building and deploying AI models across various platforms.


---

## 8. Pillar 4 & 5 — LangGraph & LangSmith (Overview)

### LangGraph — Agents with Graph-Based Control Flow

Chains are **linear**: Step A → Step B → Step C.

But real agents need **cycles**: reason → act → observe → decide to continue or try a different approach.

**LangGraph** provides:
- **State machines** — persistent state across steps
- **Nodes** — individual processing steps (LLM calls, tools, human input)
- **Edges** — connections with conditional routing
- **Human-in-the-loop** — pause for human approval

| Feature | LangChain Chains | LangGraph |
|---------|------------------|-----------|
| Flow | Linear | Cyclical, conditional |
| State | Stateless | Stateful + persistent |
| Control | Implicit | Explicit graph |
| Use case | Simple pipelines | Complex agents |

### LangSmith — Observability & Monitoring

When an LLM app gives a wrong answer — was it the prompt? The retrieved context? The model? LangSmith provides:

- **Tracing** — see every step of every chain execution
- **Evaluation** — automated testing of LLM outputs
- **Monitoring** — track latency, cost, errors in production
- **Datasets** — curate test cases for regression testing

> LangSmith integrates automatically — just set an environment variable and every execution is traced.

---

## 9. Putting It All Together — A Multi-Step Chain

Let's build a practical example that combines prompt templates, LCEL chains, and output parsing.

### Experiment 9A: Two-Step Chain — Generate then Analyze

In [18]:
# Step 1: Generate a concept explanation
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Explain concepts clearly in 2-3 sentences."),
    ("human", "Explain: {concept}")
])

explain_chain = explain_prompt | llm | StrOutputParser()

# Step 2: Simplify the explanation for a child
simplify_prompt = ChatPromptTemplate.from_messages([
    ("system", "You simplify technical explanations for a 10-year-old. Use an analogy. 2 sentences max."),
    ("human", "Simplify this: {explanation}")
])

simplify_chain = simplify_prompt | llm | StrOutputParser()

# Run them in sequence
concept = "vector embeddings"

print(f"Concept: {concept}")
print(f"\n{'=' * 50}")
print("Step 1 — Technical Explanation:")
print(f"{'=' * 50}")

explanation = explain_chain.invoke({"concept": concept})
display(Markdown(explanation))

print(f"\n{'=' * 50}")
print("Step 2 — Simplified for a 10-year-old:")
print(f"{'=' * 50}")

simple = simplify_chain.invoke({"explanation": explanation})
display(Markdown(simple))

Concept: vector embeddings

Step 1 — Technical Explanation:


Vector embeddings refer to techniques used in natural language processing where textual data is converted into numerical vectors, which represent the meaning of words or phrases. This method helps computers understand and analyze text more effectively by capturing semantic relationships between words, enabling better performance in tasks like language translation, sentiment analysis, and machine translation.


Step 2 — Simplified for a 10-year-old:


Imagine you have a big box of colorful blocks. Some blocks are for different things - cars, houses, trees. Now think about all those blocks being put into numbered bags based on their colors. These numbers help understand what each block is even if they're in different sizes or colors. This way, when you want to build a house with your toy blocks, the computer can figure out that it's similar to building a house because both use blocks (or words) for parts of it.

### Experiment 9B: Chaining with RunnableLambda (Custom Logic in a Chain)

In [21]:
from langchain_core.runnables import RunnableLambda

# Custom function wrapped as a Runnable
def format_as_flashcard(text):
    """Format LLM output as a study flashcard."""
    lines = text.strip().split("\n")
    title = lines[0] if lines else "Concept"
    body = "\n".join(lines[1:]) if len(lines) > 1 else text
    return f"{'=' * 40}\n  FLASHCARD\n{'=' * 40}\n{text}\n{'=' * 40}"

# Build the chain: prompt → model → parse → custom format
flashcard_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Give a brief definition (1-2 sentences) then one key example."),
        ("human", "Define: {term}")
    ])
    | llm
    | StrOutputParser()
    | RunnableLambda(format_as_flashcard)  # Custom post-processing
)

terms = ["LCEL", "Prompt Template", "Output Parser"]

for term in terms:
    result = flashcard_chain.invoke({"term": term})
    print(result)
    print()

  FLASHCARD
**L CEL (Long Contextual Embeddings Learning)** is a deep learning approach used to learn contextualized embeddings for sentences or sequences of words.

The main idea behind L CEL is to use a multi-layer bidirectional neural network architecture, similar to those used in machine translation and natural language inference tasks. The network takes a sequence of tokens as input and outputs a fixed-length vector representation that captures the meaning of the entire sentence.

L CEL differs from traditional sentence embedding approaches like Sentence-BERT (sBERT) or Universal Sentence Encoder (USE), where the embedding layer is typically learned separately from the rest of the model. In contrast, L CEL uses an embedding layer as part of the neural network architecture itself, which allows for more efficient learning and better capture of contextual dependencies.

The key benefits of L CEL include:

* **Improved contextual understanding**: L CEL can learn to represent entire se

---

## 10. Package Installation Strategy

Install **only what you need**. The modular structure prevents dependency bloat.

| Use Case | Packages |
|----------|----------|
| Basic LLM apps (local) | `langchain langchain-core langchain-ollama` |
| Basic LLM apps (OpenAI) | `langchain langchain-core langchain-openai` |
| RAG applications | `langchain langchain-ollama langchain-chroma langchain-community` |
| Agent applications | `langgraph langchain-ollama` |
| Observability | `langsmith` |

```bash
# What we installed for this notebook:
pip install langchain langchain-core langchain-ollama
```

---

## 11. Sandbox — Try It Yourself!

In [ ]:
# ============================================================
#  SANDBOX - Edit and re-run!
# ============================================================

# Build your own chain
my_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep it brief."),
    ("human", "{question}")
])

my_chain = my_prompt | llm | StrOutputParser()

# Try different questions
question = "What makes LangChain different from calling an LLM API directly?"

use_streaming = True  # Toggle: True for streaming, False for invoke

# ============================================================

if use_streaming:
    print("[STREAMING]\n")
    for chunk in my_chain.stream({"question": question}):
        print(chunk, end="", flush=True)
    print()
else:
    print("[INVOKE]\n")
    result = my_chain.invoke({"question": question})
    display(Markdown(result))

---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **LangChain** | An ecosystem of 5 pillars — Core, LangChain, Community, LangGraph, LangSmith |
| **Core Abstractions** | Everything implements the Runnable protocol: `invoke()`, `stream()`, `batch()` |
| **Message Types** | `SystemMessage`, `HumanMessage`, `AIMessage` — structured, not raw dicts |
| **Prompt Templates** | Reusable, parameterized prompts with `ChatPromptTemplate` |
| **LCEL** | Compose chains with `\|` pipe operator: `prompt \| model \| parser` |
| **Output Parsers** | Convert raw LLM text to strings, lists, JSON, or custom formats |
| **Memory** | LLMs have no memory — you manage conversation history yourself |
| **Swappability** | Same chain works with any provider — change one import line |
| **Modular Install** | Install only what you need: `langchain-ollama`, `langchain-openai`, etc. |
| **LangGraph** | For stateful agents needing cycles, conditions, and persistence |
| **LangSmith** | Observability — tracing, evaluation, and monitoring for production |